# TradeSense · Component 03 · Layer 2 real-data benchmarks (notebook 39, CPU)

Two **separate** benchmark classifiers on the **frozen Layer 2 v3 splits** (nothing is re-split here):

| Source | Label | Verdict in your manifests | How it is reported |
|---|---|---|---|
| **Sapienza** pump-and-dump (5-second windows) | `gt` (pump) | READY | primary real-data evidence |
| **Mt.Gox** wash trading (trade-level) | `wash` | **EXPLORATORY** (label derivation undocumented, no time column, no participant id) | exploratory benchmark, reported with those limits |

Ardia–Bluteau (events + Binance klines) is a separate notebook (39b) because it needs a download and a new pre-registered split.

Per source: XGBoost (config chosen on **VALIDATION only**), baselines (Isolation Forest, robust z-score, logistic regression), PR-AUC / ROC-AUC with block-bootstrap CIs, class-wise precision/recall/F1 at a **validation-chosen threshold**, confusion matrix, calibration (before/after Platt fitted on VAL), SHAP, and the pre-registered hypotheses H-L1..H-L3. **The test partition is read once**, after the selection is saved.

No synthetic data is mixed in. Domain shift is stated: other assets / exchanges / eras than BTCUSDT/ETHUSDT Binance Spot, so these are *external* validation of the approach, not of the bar models themselves.

In [ ]:
import os, json, glob, time, hashlib, warnings, sys
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
F3 = BASE/'processed_layer2_final_v3'; MR = BASE/'processed_layer2_model_ready_v1'; TI = BASE/'processed_layer2_targeted_improvements_v1'
OUT = BASE/'layer2_benchmarks_v1'; OUT.mkdir(parents=True, exist_ok=True)
MOCK = os.environ.get('TS_MOCK') == '1'
NBOOT = {'sapienza': 100 if MOCK else 400, 'mtgox': 40 if MOCK else 120}
BLOCK = {'sapienza': 500, 'mtgox': 5000}
SEEDS = [11, 22, 33]
LINES = []
def say(s=''): print(s); LINES.append(str(s))
try:
    import xgboost as xgb
except ImportError:
    os.system('pip -q install xgboost'); import xgboost as xgb
try:
    import shap
except ImportError:
    os.system('pip -q install shap'); import shap
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             precision_recall_fscore_support, confusion_matrix)
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
say('xgboost ' + xgb.__version__ + ('  [MOCK RUN]' if MOCK else ''))

Mounted at /content/drive
xgboost 3.4.1


## 1 · Pre-registration (written and hashed before the test partitions are read)

In [ ]:
PROTO = """# Layer 2 benchmarks v1 (pre-registration, written before the frozen test partitions are read)
Sources: Sapienza (frozen v3 split, READY) and Mt.Gox (frozen v3 split, EXPLORATORY). Splits, features and preprocessing are the frozen v3 ones; nothing re-fitted or re-split.
Models: XGBoost (grid: depth {3,5} x scale_pos_weight {full, sqrt, none} from TRAIN labels, early stopping on VALIDATION PR-AUC, config chosen by VALIDATION PR-AUC; final = mean of seeds 11/22/33).
Baselines: Isolation Forest (fit on TRAIN rows with label 0), robust z-score (max |z| over features, median/MAD from TRAIN), logistic regression (standardised, TRAIN).
Threshold: maximum F1 on VALIDATION, applied unchanged to TEST. Calibration: Platt fitted on VALIDATION.
Intervals: block bootstrap over contiguous row blocks (Sapienza 500 rows, Mt.Gox 5000 rows), 95%.
Hypotheses:
 H-L1 (Sapienza): XGBoost TEST PR-AUC exceeds the best baseline PR-AUC by >= 0.05 and the paired block-bootstrap 95% CI of the difference is above 0.
 H-L2 (Sapienza): at the VALIDATION threshold, TEST precision >= 0.5 and TEST recall >= 0.5.
 H-L3 (Mt.Gox, exploratory): XGBoost TEST PR-AUC exceeds TEST prevalence by >= 0.10 and exceeds logistic regression by >= 0.02 with paired CI above 0.
Test discipline: selection saved to declared_layer2_v1.json BEFORE test rows are scored; test metrics computed once.
Limits stated in advance: Sapienza positives are few (test pumps ~48): wide intervals. Mt.Gox wash label has no documented derivation, rows have no time column and no participant id, and 1.2% of train+val rows sit in conflicting-label clusters: results are exploratory and are NOT evidence about manipulation on BTCUSDT/ETHUSDT. Both sources are other assets/eras than the live system (domain shift).
"""
pp = OUT/'protocol_layer2_v1.md'; ph = OUT/'protocol_layer2_v1.sha256'
if pp.exists():
    assert pp.read_text() == PROTO, 'protocol_layer2_v1.md exists and differs from this notebook: frozen protocol was changed'
else:
    pp.write_text(PROTO); ph.write_text(hashlib.sha256(PROTO.encode()).hexdigest() + '\n')
say('protocol layer2 v1 sha256 ' + hashlib.sha256(PROTO.encode()).hexdigest()[:16])

protocol layer2 v1 sha256 51d640582fa3d737


## 2 · Helpers

In [ ]:
def pr_auc(y, s): return float(average_precision_score(y, s))
def roc(y, s): return float(roc_auc_score(y, s))
def best_f1_threshold(y_val, s_val):
    p, r, t = precision_recall_curve(y_val, s_val)
    f = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
    return float(t[int(np.argmax(f))])
def at_threshold(y, s, t):
    pred = (s >= t).astype(int)
    P, R, F, _ = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(threshold=float(t), precision1=float(P[1]), recall1=float(R[1]), f1_1=float(F[1]), precision0=float(P[0]), recall0=float(R[0]), f1_0=float(F[0]),
                tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn))
def ece_quantile(y, p, nb=10):
    q = np.quantile(p, np.linspace(0, 1, nb + 1)); q[0] -= 1e-12; q[-1] += 1e-12; b = np.digitize(p, q[1:-1]); e = 0.0
    for k in range(nb):
        m = b == k
        if m.any(): e += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(e)
def reliability(y, p, nb=8):
    q = np.quantile(p, np.linspace(0, 1, nb + 1)); q[0] -= 1e-12; q[-1] += 1e-12; b = np.digitize(p, q[1:-1]); out = []
    for k in range(nb):
        m = b == k
        if m.any(): out.append((float(p[m].mean()), float(y[m].mean()), int(m.sum())))
    return out
def block_idx(n, block, rng):
    nblk = int(np.ceil(n / block)); pick = rng.integers(0, nblk, nblk)
    return np.concatenate([np.arange(b * block, min((b + 1) * block, n)) for b in pick])
def block_boot(y, scores, fn, block, nb, seed=0):
    """scores: dict name -> array. returns dict name -> (lo, hi) for fn, and paired diffs vs the first key given as 'ref' handled by caller"""
    rng = np.random.default_rng(seed); n = len(y); res = {k: [] for k in scores}
    for _ in range(nb):
        ix = block_idx(n, block, rng); yy = y[ix]
        if yy.min() == yy.max(): continue
        for k, v in scores.items(): res[k].append(fn(yy, v[ix]))
    return {k: np.array(v) for k, v in res.items()}
def ci(a): return (float(np.quantile(a, 0.025)), float(np.quantile(a, 0.975)))
def robust_z_fit(X): mu = np.median(X, 0); sc = np.median(np.abs(X - mu), 0) * 1.4826 + 1e-9; return mu, sc
def robust_z_score(X, mu, sc): return np.abs((X - mu) / sc).max(1)
def spw_options(y):
    full = float((y == 0).sum() / max((y == 1).sum(), 1)); return {'full': full, 'sqrt': float(np.sqrt(full)), 'none': 1.0}
def fit_xgb(Xtr, ytr, Xva, yva, depth, spw, seed):
    m = xgb.XGBClassifier(n_estimators=300 if MOCK else 700, learning_rate=0.06, max_depth=depth, subsample=0.8, colsample_bytree=0.8, min_child_weight=1,
                          scale_pos_weight=spw, eval_metric='aucpr', early_stopping_rounds=40, random_state=seed, n_jobs=-1, tree_method='hist')
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False); return m

## 3 · Frozen Layer 2 v3 data (checked against your manifests)

In [ ]:
SRC = {}
def load_source(name):
    if name == 'sapienza':
        man = json.load(open(MR/'sapienza'/'training_manifest.json')); feats = man['features']['order']; tgt = man['target_column']
        files = {k: F3/f'sapienza_{k}.parquet' for k in ['train', 'val', 'test']}
    else:
        man = json.load(open(MR/'mtgox'/'training_manifest.json')); feats = man['features']['order']; tgt = man['target_column']
        files = {k: F3/f'mtgox_{k}.parquet' for k in ['train', 'val', 'test']}
    d = {}
    for k, p in files.items():
        assert p.exists(), f'{p} missing'
        df = pd.read_parquet(p)
        assert all(c in df.columns for c in feats + [tgt]), f'{p.name}: columns do not match the manifest'
        d[k] = df
    return dict(man=man, feats=feats, tgt=tgt, files=files, d=d)
for nm in ['sapienza', 'mtgox']:
    SRC[nm] = load_source(nm)
    S = SRC[nm]; say(f'{nm}: verdict {S["man"].get("verdict")}, {len(S["feats"])} features, target {S["tgt"]}')
    for k in ['train', 'val', 'test']:
        df = S['d'][k]; say(f'   {k}: {len(df):,} rows, positives {int(df[S["tgt"]].sum()):,} ({df[S["tgt"]].mean():.4%})')

sapienza: verdict READY, 15 features, target gt
   train: 600,827 rows, positives 220 (0.0366%)
   val: 121,255 rows, positives 44 (0.0363%)
   test: 89,334 rows, positives 48 (0.0537%)
mtgox: verdict EXPLORATORY, 7 features, target wash
   train: 3,876,076 rows, positives 1,525,207 (39.3493%)
   val: 830,588 rows, positives 203,976 (24.5580%)
   test: 830,588 rows, positives 173,463 (20.8844%)


## 4 · Unit tests

In [ ]:
# T01 metrics against a plain computation
_y = np.array([0, 0, 1, 1, 0, 1]); _s = np.array([.1, .4, .35, .8, .2, .9])
assert abs(roc(_y, _s) - np.mean([(a > b) + 0.5 * (a == b) for a in _s[_y == 1] for b in _s[_y == 0]])) < 1e-12; say('PASS T01 ROC-AUC equals the pairwise definition')
# T02 threshold depends only on the validation arrays
_t1 = best_f1_threshold(_y, _s); _t2 = best_f1_threshold(_y.copy(), _s.copy()); assert _t1 == _t2; say('PASS T02 threshold is a function of validation arrays only')
# T03 split sizes and positives equal the manifests (frozen)
for nm in ['sapienza', 'mtgox']:
    S = SRC[nm]
    if nm == 'mtgox':
        for k in ['train', 'val', 'test']:
            if MOCK: continue
            sp = S['man']['splits'][k]; df = S['d'][k]
            assert len(df) == sp['rows'] and int(df[S['tgt']].sum()) == sp['positives'], f'mtgox {k} differs from the manifest'
    else:
        ar = json.load(open(MR/'sapienza'/'reports'/'alignment_report.json')) if not MOCK else None
        if ar:
            for k in ['train', 'val', 'test']: assert int(S['d'][k][S['tgt']].sum()) == ar['positive_rows'][k], f'sapienza {k} positives differ'
say('PASS T03 row and positive counts equal the frozen manifests' + (' (skipped in mock)' if MOCK else ''))
# T04 frozen files unchanged (sha256 of the parquets against the manifest / pins)
if not MOCK:
    pins = json.load(open(TI/'sapienza'/'training_manifest_addendum.json'))['frozen_v3_inputs_pin']
    for k in ['train', 'val', 'test']:
        got = hashlib.sha256(SRC['sapienza']['files'][k].read_bytes()).hexdigest(); assert got == pins[f'sapienza_{k}.parquet'], f'sapienza_{k}.parquet changed'
        exp = SRC['mtgox']['man']['splits'][k]['sha256']; got = hashlib.sha256(SRC['mtgox']['files'][k].read_bytes()).hexdigest(); assert got == exp, f'mtgox_{k}.parquet changed'
    say('PASS T04 all six frozen parquet files match their recorded sha256')
# T05 scale_pos_weight uses TRAIN labels only
_o = spw_options(SRC['mtgox']['d']['train'][SRC['mtgox']['tgt']].values); assert abs(_o['full'] - (1 / (SRC['mtgox']['d']['train'][SRC['mtgox']['tgt']].mean()) - 1)) < 1e-6; say('PASS T05 class weights come from the TRAIN labels')
# T06 block bootstrap interval brackets the point estimate
_rng = np.random.default_rng(1); _yy = (_rng.random(4000) < 0.2).astype(int); _ss = _yy * 0.5 + _rng.random(4000)
_bb = block_boot(_yy, {'a': _ss}, pr_auc, 200, 100); _lo, _hi = ci(_bb['a']); assert _lo <= pr_auc(_yy, _ss) <= _hi; say('PASS T06 block-bootstrap interval brackets the point estimate')

PASS T01 ROC-AUC equals the pairwise definition
PASS T02 threshold is a function of validation arrays only
PASS T03 row and positive counts equal the frozen manifests
PASS T04 all six frozen parquet files match their recorded sha256
PASS T05 class weights come from the TRAIN labels
PASS T06 block-bootstrap interval brackets the point estimate


## 5 · Selection on VALIDATION only (saved before test)

In [ ]:
RES = {}
for nm in ['sapienza', 'mtgox']:
    S = SRC[nm]; f, t = S['feats'], S['tgt']
    Xtr, ytr = S['d']['train'][f].values.astype(np.float32), S['d']['train'][t].values.astype(int)
    Xva, yva = S['d']['val'][f].values.astype(np.float32), S['d']['val'][t].values.astype(int)
    opts = spw_options(ytr); table = []
    for depth in [3, 5]:
        for wn, w in opts.items():
            t0 = time.time(); m = fit_xgb(Xtr, ytr, Xva, yva, depth, w, 11); pv = m.predict_proba(Xva)[:, 1]
            table.append(dict(depth=depth, spw=wn, spw_value=w, best_iter=int(m.best_iteration), val_prauc=pr_auc(yva, pv), val_roc=roc(yva, pv), secs=round(time.time() - t0, 1)))
            print(nm, table[-1], flush=True)
    tb = pd.DataFrame(table).sort_values('val_prauc', ascending=False).reset_index(drop=True)
    best = tb.iloc[0]; say(f'--- {nm}: validation selection (seed 11) ---'); say(tb.round(4).to_string(index=False))
    RES[nm] = dict(table=tb, depth=int(best.depth), spw_name=str(best.spw), spw=float(best.spw_value))
    tb.to_csv(OUT/f'select_{nm}.csv', index=False)
declared = {nm: dict(depth=RES[nm]['depth'], scale_pos_weight=RES[nm]['spw'], spw_name=RES[nm]['spw_name']) for nm in RES}
json.dump(declared, open(OUT/'declared_layer2_v1.json', 'w'), indent=1); say('declared_layer2_v1.json saved BEFORE any test row is scored: ' + json.dumps(declared))

sapienza {'depth': 3, 'spw': 'full', 'spw_value': 2730.0318181818184, 'best_iter': 177, 'val_prauc': 0.9323404192607806, 'val_roc': 0.9951030172029092, 'secs': 8.2}
sapienza {'depth': 3, 'spw': 'sqrt', 'spw_value': 52.24970639325946, 'best_iter': 91, 'val_prauc': 0.9359287596288842, 'val_roc': 0.9996722469682845, 'secs': 7.2}
sapienza {'depth': 3, 'spw': 'none', 'spw_value': 1.0, 'best_iter': 66, 'val_prauc': 0.9472634988495996, 'val_roc': 0.9998398735188301, 'secs': 4.1}
sapienza {'depth': 5, 'spw': 'full', 'spw_value': 2730.0318181818184, 'best_iter': 253, 'val_prauc': 0.944140554109331, 'val_roc': 0.999674871992566, 'secs': 13.6}
sapienza {'depth': 5, 'spw': 'sqrt', 'spw_value': 52.24970639325946, 'best_iter': 296, 'val_prauc': 0.9481659578058004, 'val_roc': 0.9998738113327549, 'secs': 16.5}
sapienza {'depth': 5, 'spw': 'none', 'spw_value': 1.0, 'best_iter': 44, 'val_prauc': 0.9452170257514783, 'val_roc': 0.9997181848932103, 'secs': 3.3}
--- sapienza: validation selection (seed 11) 

## 6 · Final fit and the single test scoring

In [ ]:
declared = json.load(open(OUT/'declared_layer2_v1.json')); FINAL = {}
for nm in ['sapienza', 'mtgox']:
    S = SRC[nm]; f, t = S['feats'], S['tgt']; dcl = declared[nm]
    Xtr, ytr = S['d']['train'][f].values.astype(np.float32), S['d']['train'][t].values.astype(int)
    Xva, yva = S['d']['val'][f].values.astype(np.float32), S['d']['val'][t].values.astype(int)
    Xte, yte = S['d']['test'][f].values.astype(np.float32), S['d']['test'][t].values.astype(int)
    models = [fit_xgb(Xtr, ytr, Xva, yva, dcl['depth'], dcl['scale_pos_weight'], sd) for sd in SEEDS]
    pva = np.mean([m.predict_proba(Xva)[:, 1] for m in models], 0); pte = np.mean([m.predict_proba(Xte)[:, 1] for m in models], 0)
    # baselines (TRAIN only)
    iso = IsolationForest(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr[ytr == 0]); sva_if, ste_if = -iso.score_samples(Xva), -iso.score_samples(Xte)
    mu, sc = robust_z_fit(Xtr); sva_rz, ste_rz = robust_z_score(Xva, mu, sc), robust_z_score(Xte, mu, sc)
    ss = StandardScaler().fit(Xtr); lr = LogisticRegression(max_iter=300, C=1.0, class_weight='balanced').fit(ss.transform(Xtr), ytr)
    sva_lr, ste_lr = lr.predict_proba(ss.transform(Xva))[:, 1], lr.predict_proba(ss.transform(Xte))[:, 1]
    FINAL[nm] = dict(models=models, yva=yva, yte=yte, Xte=Xte, pva=pva, pte=pte, base_te={'IsolationForest': ste_if, 'robust z-score': ste_rz, 'LogReg': ste_lr}, base_va={'IsolationForest': sva_if, 'robust z-score': sva_rz, 'LogReg': sva_lr})
    say(f'{nm}: test scored once ({len(yte):,} rows, {int(yte.sum()):,} positives, prevalence {yte.mean():.4%})')

sapienza: test scored once (89,334 rows, 48 positives, prevalence 0.0537%)
mtgox: test scored once (830,588 rows, 173,463 positives, prevalence 20.8844%)


## 7 · Test results, baselines, threshold, confusion matrix, calibration, hypotheses

In [ ]:
HYP = {}; ROWS = []
for nm in ['sapienza', 'mtgox']:
    F = FINAL[nm]; yte, pte, yva, pva = F['yte'], F['pte'], F['yva'], F['pva']; prev = float(yte.mean())
    allsc = {'XGBoost': pte, **F['base_te']}
    bb = block_boot(yte, allsc, pr_auc, BLOCK[nm], NBOOT[nm], seed=3); bbr = block_boot(yte, allsc, roc, BLOCK[nm], NBOOT[nm], seed=3)
    rows = []
    for k, v in allsc.items():
        rows.append(dict(source=nm, model=k, test_prauc=pr_auc(yte, v), prauc_ci='%.3f-%.3f' % ci(bb[k]), test_roc=roc(yte, v), roc_ci='%.3f-%.3f' % ci(bbr[k]), prevalence=prev, prauc_over_prevalence=pr_auc(yte, v) / prev))
    tbl = pd.DataFrame(rows); ROWS.append(tbl)
    say(f'--- {nm}: TEST discrimination (block-bootstrap 95% CI) ---'); say(tbl.round(4).to_string(index=False))
    # operating point from VALIDATION
    thr = best_f1_threshold(yva, pva); op = at_threshold(yte, pte, thr); opv = at_threshold(yva, pva, thr)
    say(f'{nm}: threshold from VALIDATION (max F1) = {thr:.4f};  VALIDATION P/R/F1 {opv["precision1"]:.3f}/{opv["recall1"]:.3f}/{opv["f1_1"]:.3f}')
    say(f'{nm}: TEST class 1 precision {op["precision1"]:.3f} recall {op["recall1"]:.3f} F1 {op["f1_1"]:.3f} | class 0 precision {op["precision0"]:.4f} recall {op["recall0"]:.4f} F1 {op["f1_0"]:.4f}')
    say(f'{nm}: TEST confusion matrix  TN {op["tn"]:,}  FP {op["fp"]:,}  FN {op["fn"]:,}  TP {op["tp"]:,}')
    # calibration (Platt on VALIDATION)
    lv = np.log(np.clip(pva, 1e-6, 1 - 1e-6) / (1 - np.clip(pva, 1e-6, 1 - 1e-6))).reshape(-1, 1); lt = np.log(np.clip(pte, 1e-6, 1 - 1e-6) / (1 - np.clip(pte, 1e-6, 1 - 1e-6))).reshape(-1, 1)
    pl = LogisticRegression(C=1e6, max_iter=500).fit(lv, yva); pte_c = pl.predict_proba(lt)[:, 1]
    say(f'{nm}: calibration (quantile-bin ECE, TEST): raw {ece_quantile(yte, pte):.4f} -> Platt-on-VAL {ece_quantile(yte, pte_c):.4f}; reliability (mean p, observed rate, n): ' + str([(round(a, 3), round(b, 3), c) for a, b, c in reliability(yte, pte_c)]))
    # hypotheses
    best_base = max(F['base_te'], key=lambda k: pr_auc(yte, F['base_te'][k]))
    diff = bb['XGBoost'] - bb[best_base]; dlo, dhi = ci(diff); dpt = pr_auc(yte, pte) - pr_auc(yte, F['base_te'][best_base])
    if nm == 'sapienza':
        HYP['H-L1'] = dict(best_baseline=best_base, difference=dpt, ci=[dlo, dhi], holds=bool(dpt >= 0.05 and dlo > 0))
        HYP['H-L2'] = dict(precision=op['precision1'], recall=op['recall1'], holds=bool(op['precision1'] >= 0.5 and op['recall1'] >= 0.5))
        PREV_S = prev
    else:
        dl = bb['XGBoost'] - bb['LogReg']; llo, lhi = ci(dl); dlp = pr_auc(yte, pte) - pr_auc(yte, F['base_te']['LogReg'])
        HYP['H-L3 (exploratory)'] = dict(xgb_prauc=pr_auc(yte, pte), prevalence=prev, over_prevalence=pr_auc(yte, pte) - prev, vs_logreg=dlp, ci_vs_logreg=[llo, lhi], holds=bool(pr_auc(yte, pte) - prev >= 0.10 and dlp >= 0.02 and llo > 0))
    FINAL[nm]['op'] = op; FINAL[nm]['thr'] = thr
say('\n--- pre-registered hypotheses ---')
for k, v in HYP.items(): say(k + ': ' + json.dumps({a: (round(b, 4) if isinstance(b, float) else b) for a, b in v.items()}))
pd.concat(ROWS).to_csv(OUT/'layer2_benchmark_test.csv', index=False)

--- sapienza: TEST discrimination (block-bootstrap 95% CI) ---
  source           model  test_prauc    prauc_ci  test_roc      roc_ci  prevalence  prauc_over_prevalence
sapienza         XGBoost      0.9146 0.867-0.963    0.9999 1.000-1.000      0.0005              1702.2406
sapienza IsolationForest      0.5038 0.388-0.623    0.9981 0.996-0.999      0.0005               937.6048
sapienza  robust z-score      0.2930 0.194-0.421    0.9875 0.974-0.995      0.0005               545.3128
sapienza          LogReg      0.4615 0.361-0.572    0.9788 0.925-1.000      0.0005               858.9007
sapienza: threshold from VALIDATION (max F1) = 0.9401;  VALIDATION P/R/F1 1.000/0.864/0.927
sapienza: TEST class 1 precision 0.857 recall 0.750 F1 0.800 | class 0 precision 0.9999 recall 0.9999 F1 0.9999
sapienza: TEST confusion matrix  TN 89,280  FP 6  FN 12  TP 36
sapienza: calibration (quantile-bin ECE, TEST): raw 0.0003 -> Platt-on-VAL 0.0003; reliability (mean p, observed rate, n): [(0.0, 0.0, 55834

## 8 · SHAP

In [ ]:
SHAPV = {}
for nm in ['sapienza', 'mtgox']:
    F = FINAL[nm]; S = SRC[nm]; rng = np.random.default_rng(0); yte, Xte = F['yte'], F['Xte']
    pos = np.where(yte == 1)[0]; neg = np.where(yte == 0)[0]
    ix = np.concatenate([rng.choice(pos, min(len(pos), 2000), replace=False), rng.choice(neg, min(len(neg), 4000 if not MOCK else 500), replace=False)])
    sv = shap.TreeExplainer(F['models'][0]).shap_values(Xte[ix]); sv = sv[1] if isinstance(sv, list) else sv
    imp = pd.Series(np.abs(sv).mean(0), index=S['feats']).sort_values(ascending=False); SHAPV[nm] = imp
    say(f'--- {nm}: mean |SHAP| (seed-11 model, {len(ix):,} test rows: all-positives sample + negatives) ---'); say(imp.round(4).to_string())
    imp.to_csv(OUT/f'shap_{nm}.csv')

--- sapienza: mean |SHAP| (seed-11 model, 4,048 test rows: all-positives sample + negatives) ---
std_rush_order             2.4508
std_trades                 1.6443
avg_rush_order             1.2643
minute_sin                 0.7815
hour_sin                   0.6562
avg_volume                 0.4977
minute_cos                 0.3789
std_price                  0.2720
rush_volume_interaction    0.2620
hour_cos                   0.2520
avg_price                  0.2519
avg_price_max              0.2073
log_avg_volume             0.1588
std_volume                 0.0425
log_std_price              0.0276
--- mtgox: mean |SHAP| (seed-11 model, 6,000 test rows: all-positives sample + negatives) ---
buyer_24h_trade_count     0.5693
seller_24h_trade_count    0.3768
abs_price_deviation       0.1107
trade_count_ratio         0.0558
log_bitcoins              0.0457
bitcoins                  0.0226
price_deviation           0.0079


## 9 · Save

In [ ]:
res = dict(protocol_sha=hashlib.sha256(PROTO.encode()).hexdigest(), declared=declared, hypotheses=HYP,
           test={nm: dict(op=FINAL[nm]['op'], thr=FINAL[nm]['thr'], prevalence=float(FINAL[nm]['yte'].mean()), n=len(FINAL[nm]['yte']), positives=int(FINAL[nm]['yte'].sum())) for nm in FINAL})
json.dump(res, open(OUT/'layer2_results_v1.json', 'w'), indent=1, default=float)
say('\nReading per source:')
say('Sapienza (READY): real pump-and-dump labels, group-disjoint chronological split, 1h embargo; evidence that the manipulation-detection approach works on real labelled pumps. Domain shift: altcoin/BTC pairs 2018-2021, 5-second windows.')
say('Mt.Gox (EXPLORATORY): wash-trading label with undocumented derivation, no time column; reported as an exploratory benchmark only, not as evidence about BTCUSDT/ETHUSDT.')
(OUT/'b2_step10_share_block.txt').write_text('\n'.join(LINES)); say('saved ' + str(OUT/'b2_step10_share_block.txt'))


Reading per source:
Sapienza (READY): real pump-and-dump labels, group-disjoint chronological split, 1h embargo; evidence that the manipulation-detection approach works on real labelled pumps. Domain shift: altcoin/BTC pairs 2018-2021, 5-second windows.
Mt.Gox (EXPLORATORY): wash-trading label with undocumented derivation, no time column; reported as an exploratory benchmark only, not as evidence about BTCUSDT/ETHUSDT.
saved /content/drive/MyDrive/research datasets/layer2_benchmarks_v1/b2_step10_share_block.txt


## 10 · Share block

In [ ]:
print('\n===== SHARE BLOCK (attach layer2_benchmarks_v1/b2_step10_share_block.txt, or copy below) =====')
print('\n'.join(LINES))


===== SHARE BLOCK (attach layer2_benchmarks_v1/b2_step10_share_block.txt, or copy below) =====
xgboost 3.4.1
protocol layer2 v1 sha256 51d640582fa3d737
sapienza: verdict READY, 15 features, target gt
   train: 600,827 rows, positives 220 (0.0366%)
   val: 121,255 rows, positives 44 (0.0363%)
   test: 89,334 rows, positives 48 (0.0537%)
mtgox: verdict EXPLORATORY, 7 features, target wash
   train: 3,876,076 rows, positives 1,525,207 (39.3493%)
   val: 830,588 rows, positives 203,976 (24.5580%)
   test: 830,588 rows, positives 173,463 (20.8844%)
PASS T01 ROC-AUC equals the pairwise definition
PASS T02 threshold is a function of validation arrays only
PASS T03 row and positive counts equal the frozen manifests
PASS T04 all six frozen parquet files match their recorded sha256
PASS T05 class weights come from the TRAIN labels
PASS T06 block-bootstrap interval brackets the point estimate
--- sapienza: validation selection (seed 11) ---
 depth  spw  spw_value  best_iter  val_prauc  val_roc  